# TSTR CTGAN Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')
# os.makedirs('RESULTS', exist_ok=True)

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)

#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(SYN_DATA_HOME + '1_Chap_Data_Synthetic_CTGAN.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,6.094322,-0.208446,0.416890,0.515579,-7.706072,-0.527509,0.166410,0.211860
1,Group0,2.291828,-0.762772,-0.112074,0.083636,0.846565,1.539027,-0.004124,0.277265
2,Group0,-5.112419,-0.377207,0.224469,0.505163,7.457264,-0.497955,-0.752442,0.063517
3,Group0,6.007161,-0.165590,-0.152581,0.389270,-4.636239,-0.214006,-0.530647,0.046882
4,Group0,1.082027,-1.209130,-0.070217,0.204933,1.570072,0.721873,0.099834,0.010630
...,...,...,...,...,...,...,...,...,...
2712,Group0,3.814451,0.190931,-0.282501,0.325604,-2.521737,-0.304934,-0.905134,-0.061680
2713,Group0,-1.059330,0.216962,-0.006728,0.091072,-5.399747,1.604053,-1.166298,-0.027417
2714,Group0,9.769922,0.944167,0.407352,0.002913,-4.778331,0.129506,0.001454,0.194900
2715,Group0,-4.384517,0.040398,-0.064242,0.055481,-5.280489,-0.847199,0.277913,0.162433


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
# print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'
categorical_columns = []
numerical_columns = train_data.select_dtypes(include=['int64','float64']).columns.tolist()
categories = [np.array(range(2))] if categorical_columns else []
data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(train_data.loc[:, train_data.columns != target])
y_train = train_data.loc[:, target]

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.1s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.0s finished


,model,accuracy,precision,recall,f1
0,RF,0.9043,0.9457,0.9043,0.9207


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,0.9146,0.944,0.9146,0.9266


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,0.8689,0.9414,0.8689,0.8975


## 7. Train and evaluate Support Vector Machines Classifier

In [12]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

[LibSVM]WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -526.970590, rho = 0.715104
nSV = 203, nBSV = 0
Total nSV = 203
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -495.048645, rho = 0.043378
nSV = 217, nBSV = 0
Total nSV = 217
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -487.298068, rho = 0.529119
nSV = 215, nBSV = 0
Total nSV = 215
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -439.985243, rho = 0.197008
nSV = 238, nBSV = 0
Total nSV = 238
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -456.088869, rho = 0.178670
nSV = 212, nBSV = 0
Total nSV = 212
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -376.368904, rho = 0.064034
nSV = 214, nBSV = 0
Total nSV = 214


,model,accuracy,precision,recall,f1
0,SVM,0.7511,0.9394,0.7511,0.8204


## 8. Train and evaluate Multilayer Perceptron Classifier

In [13]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results =pd.concat([results, mlp_results], ignore_index=True)
mlp_results

Iteration 1, loss = 0.63454400
Iteration 2, loss = 0.42127418
Iteration 3, loss = 0.25094303
Iteration 4, loss = 0.21312023
Iteration 5, loss = 0.20903565
Iteration 6, loss = 0.20454860
Iteration 7, loss = 0.20108483
Iteration 8, loss = 0.19905786
Iteration 9, loss = 0.19767279
Iteration 10, loss = 0.19582942
Iteration 11, loss = 0.19421170
Iteration 12, loss = 0.19163916
Iteration 13, loss = 0.19028252
Iteration 14, loss = 0.18938023
Iteration 15, loss = 0.19060224
Iteration 16, loss = 0.18865163
Iteration 17, loss = 0.18669260
Iteration 18, loss = 0.18358217
Iteration 19, loss = 0.18468438
Iteration 20, loss = 0.18448822
Iteration 21, loss = 0.18098251
Iteration 22, loss = 0.18266416
Iteration 23, loss = 0.18048336
Iteration 24, loss = 0.18007969
Iteration 25, loss = 0.17405755
Iteration 26, loss = 0.17354116
Iteration 27, loss = 0.17270955
Iteration 28, loss = 0.16986748
Iteration 29, loss = 0.17059264
Iteration 30, loss = 0.16911266
Iteration 31, loss = 0.16515180
Iteration 32, los

,model,accuracy,precision,recall,f1
0,MLP,0.8748,0.9438,0.8748,0.9017


## 9. Save results file

In [14]:
results.to_csv('RESULTS/models_results_ctgan.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,0.9043,0.9457,0.9043,0.9207
1,KNN,0.9146,0.9440,0.9146,0.9266
2,DT,0.8689,0.9414,0.8689,0.8975
3,SVM,0.7511,0.9394,0.7511,0.8204
4,MLP,0.8748,0.9438,0.8748,0.9017
